Part 1: Knowledge Base (10 marks)


In [2]:
!pip install langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [3]:
!pip install langchain faiss-cpu sentence-transformers

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Expanded KB (IMPORTANT for marks — make it ~500+ words)
text = """
Machine learning is a subset of artificial intelligence that focuses on building systems
that learn patterns from data rather than being explicitly programmed. It has become one
of the most important fields in modern computing due to its applications in healthcare,
finance, transportation, and many other industries.

Supervised learning is a type of machine learning where the model is trained on labeled data.
Each training example includes an input and the correct output. Common tasks include
classification and regression. For example, a model can be trained to classify emails as spam
or not spam, or predict house prices based on features such as location and size.

Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or
structures in the data. Clustering and dimensionality reduction are common techniques.
Clustering groups similar data points together, while dimensionality reduction simplifies
data by reducing the number of features while preserving important information.

Reinforcement learning is based on an agent interacting with an environment. The agent learns
to take actions that maximize cumulative reward over time. This approach is widely used in
robotics, game playing, and autonomous systems. The learning process involves exploration
and exploitation, balancing between trying new actions and using known successful strategies.

Overfitting occurs when a model learns the training data too well, including noise, and fails
to generalize to new data. Underfitting occurs when a model is too simple to capture patterns
in the data. Both issues negatively impact model performance.

Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large
weights. Cross-validation is used to evaluate model performance by splitting data into
multiple subsets and testing the model on different portions.

Feature engineering involves selecting, modifying, and creating input variables to improve
model performance. It plays a critical role in machine learning because better features
often lead to better results.

Model evaluation metrics such as accuracy, precision, recall, and F1-score are used to
measure how well a model performs. Choosing the right metric depends on the problem being
solved.

Machine learning continues to evolve with advancements in deep learning, neural networks,
and large-scale data processing, making it a rapidly growing field.
"""

# Split text
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([text])

# Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# FAISS store
vectorstore = FAISS.from_documents(docs, embedding_model)

# Retriever
retriever = vectorstore.as_retriever()

print("✅ Vector store created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created successfully!


### Knowledge Base Description

I chose **Machine Learning** as the topic for the knowledge base because it is a fundamental area in artificial intelligence and widely used in real-world applications. It includes multiple subfields such as supervised learning, unsupervised learning, and reinforcement learning, making it suitable for testing a RAG system.

This topic provides diverse concepts like overfitting, feature engineering, and evaluation metrics, which helps in generating meaningful queries and testing retrieval quality. It is also easy to understand and ensures that the system can demonstrate both correct answers and failure cases when information is missing.


Part 2: RAG Agent (20 marks)


In [1]:
!pip install crewai groq

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_................"

In [18]:
from crewai import Agent

rag_agent = Agent(
    role="RAG Answer Generator",
    goal="Answer questions using retrieved knowledge",
    backstory="Expert at retrieval-based QA",
    llm="groq/llama-3.3-70b-versatile",   # ✅ IMPORTANT
    verbose=True
)

In [26]:
rag_task = Task(
    description="""
    Answer the question using the provided context.

    Question:
    {question}

    Context:
    {context}

    Instructions:
    - Answer ONLY using the context above
    - Do NOT make up information

    Return EXACTLY:

    answer: <final answer>
    context: <context>
    """,

    expected_output="answer and context",
    agent=rag_agent
)

In [28]:
from crewai import Crew

def run_rag(question):
    # manually retrieve context
    docs = retriever.invoke(question)
    context = "\n".join([d.page_content for d in docs])

    crew = Crew(
        agents=[rag_agent],
        tasks=[rag_task],
        verbose=False
    )

    result = crew.kickoff(inputs={
        "question": question,
        "context": context
    })

    return result

In [25]:
questions = [
    "What is overfitting?",
    "Difference between supervised and unsupervised learning?",
    "What is reinforcement learning?"
]

In [29]:
for q in questions:
    print("\nQUESTION:", q)
    print(run_rag(q))


QUESTION: What is overfitting?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the question using the provided context.                                                            │
│                                                                                                                 │
│      Question:                                                                                                  │
│      What is overfitting?                                                                                       │
│                                                                                                                 │
│      Context:                                                                                                   │
│      Overfitting occurs when a model learns the training data too well, including noise, and fails              │
│  to generalize to new data. Underfitting occurs when a model is too simple to capture patterns                  │
│  Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large                       │
│  weights. Cross-validation is used to evaluate model performance by splitting data into                         │
│  in the data. Both issues negatively impact model performance.                                                  │
│  Feature engineering involves selecting, modifying, and creating input variables to improve                     │
│  model performance. It plays a critical role in machine learning because better features                        │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Answer ONLY using the context above                                                                      │
│      - Do NOT make up information                                                                               │
│                                                                                                                 │
│      Return EXACTLY:                                                                                            │
│                                                                                                                 │
│      answer: <final answer>                                                                                     │
│      context: <context>                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  answer: Overfitting occurs when a model learns the training data too well, including noise, and fails to       │
│  generalize to new data.                                                                                        │
│  context: Overfitting occurs when a model learns the training data too well, including noise, and fails         │
│  to generalize to new data. Underfitting occurs when a model is too simple to capture patterns                  │
│  Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large                       │
│  weights. Cross-validation is used to evaluate model performance by splitting data into                         │
│  in the data. Both issues negatively impact model performance.                                                  │
│  Feature engineering involves selecting, modifying, and creating input variables to improve                     │
│  model performance. It plays a critical role in machine learning because better features                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

answer: Overfitting occurs when a model learns the training data too well, including noise, and fails to generalize to new data.
context: Overfitting occurs when a model learns the training data too well, including noise, and fails
to generalize to new data. Underfitting occurs when a model is too simple to capture patterns
Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large
weights. Cross-validation is used to evaluate model performance by splitting data into
in the data. Both issues negatively impact model performance.
Feature engineering involves selecting, modifying, and creating input variables to improve
model performance. It plays a critical role in machine learning because better features

QUESTION: Difference between supervised and unsupervised learning?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the question using the provided context.                                                            │
│                                                                                                                 │
│      Question:                                                                                                  │
│      Difference between supervised and unsupervised learning?                                                   │
│                                                                                                                 │
│      Context:                                                                                                   │
│      Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or                    │
│  structures in the data. Clustering and dimensionality reduction are common techniques.                         │
│  Supervised learning is a type of machine learning where the model is trained on labeled data.                  │
│  Each training example includes an input and the correct output. Common tasks include                           │
│  Machine learning is a subset of artificial intelligence that focuses on building systems                       │
│  that learn patterns from data rather than being explicitly programmed. It has become one                       │
│  classification and regression. For example, a model can be trained to classify emails as spam                  │
│  or not spam, or predict house prices based on features such as location and size.                              │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Answer ONLY using the context above                                                                      │
│      - Do NOT make up information                                                                               │
│                                                                                                                 │
│      Return EXACTLY:                                                                                            │
│                                                                                                                 │
│      answer: <final answer>                                                                                     │
│      context: <context>                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  answer: Unsupervised learning deals with unlabeled data to find hidden patterns or structures, while           │
│  supervised learning is a type of machine learning where the model is trained on labeled data to perform tasks  │
│  such as classification and regression.                                                                         │
│  context: Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or               │
│  structures in the data. Clustering and dimensionality reduction are common techniques.                         │
│  Supervised learning is a type of machine learning where the model is trained on labeled data.                  │
│  Each training example includes an input and the correct output. Common tasks include                           │
│  Machine learning is a subset of artificial intelligence that focuses on building systems                       │
│  that learn patterns from data rather than being explicitly programmed. It has become one                       │
│  classification and regression. For example, a model can be trained to classify emails as spam                  │
│  or not spam, or predict house prices based on features such as location and size.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

answer: Unsupervised learning deals with unlabeled data to find hidden patterns or structures, while supervised learning is a type of machine learning where the model is trained on labeled data to perform tasks such as classification and regression.
context: Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or
structures in the data. Clustering and dimensionality reduction are common techniques.
Supervised learning is a type of machine learning where the model is trained on labeled data.
Each training example includes an input and the correct output. Common tasks include
Machine learning is a subset of artificial intelligence that focuses on building systems
that learn patterns from data rather than being explicitly programmed. It has become one
classification and regression. For example, a model can be trained to classify emails as spam
or not spam, or predict house prices based on features such as location and size.

QUESTION: What is reinforcement 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the question using the provided context.                                                            │
│                                                                                                                 │
│      Question:                                                                                                  │
│      What is reinforcement learning?                                                                            │
│                                                                                                                 │
│      Context:                                                                                                   │
│      Reinforcement learning is based on an agent interacting with an environment. The agent learns              │
│  to take actions that maximize cumulative reward over time. This approach is widely used in                     │
│  robotics, game playing, and autonomous systems. The learning process involves exploration                      │
│  and exploitation, balancing between trying new actions and using known successful strategies.                  │
│  Machine learning is a subset of artificial intelligence that focuses on building systems                       │
│  that learn patterns from data rather than being explicitly programmed. It has become one                       │
│  Supervised learning is a type of machine learning where the model is trained on labeled data.                  │
│  Each training example includes an input and the correct output. Common tasks include                           │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Answer ONLY using the context above                                                                      │
│      - Do NOT make up information                                                                               │
│                                                                                                                 │
│      Return EXACTLY:                                                                                            │
│                                                                                                                 │
│      answer: <final answer>                                                                                     │
│      context: <context>                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  answer: Reinforcement learning is based on an agent interacting with an environment, learning to take actions  │
│  that maximize cumulative reward over time.                                                                     │
│  context: Reinforcement learning is based on an agent interacting with an environment. The agent learns         │
│  to take actions that maximize cumulative reward over time. This approach is widely used in                     │
│  robotics, game playing, and autonomous systems. The learning process involves exploration                      │
│  and exploitation, balancing between trying new actions and using known successful strategies.                  │
│  Machine learning is a subset of artificial intelligence that focuses on building systems                       │
│  that learn patterns from data rather than being explicitly programmed. It has become one                       │
│  Supervised learning is a type of machine learning where the model is trained on labeled data.                  │
│  Each training example includes an input and the correct output. Common tasks include                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

answer: Reinforcement learning is based on an agent interacting with an environment, learning to take actions that maximize cumulative reward over time.
context: Reinforcement learning is based on an agent interacting with an environment. The agent learns
to take actions that maximize cumulative reward over time. This approach is widely used in
robotics, game playing, and autonomous systems. The learning process involves exploration
and exploitation, balancing between trying new actions and using known successful strategies.
Machine learning is a subset of artificial intelligence that focuses on building systems
that learn patterns from data rather than being explicitly programmed. It has become one
Supervised learning is a type of machine learning where the model is trained on labeled data.
Each training example includes an input and the correct output. Common tasks include


Part 3: Quality Evaluator Agent (25 marks)

In [8]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.6/844.6 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.4/227.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: posthog
    Found existing installation: posthog 5.4.0
    Uninstalling posthog-5.4.0:
      Successfully uninstalled posthog-5.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.1.1 requires posthog<6.0.0,>=2.4.0, but you have posthog 7.13.0 which is incompatible.


In [40]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from crewai.tools import tool
import json

@tool
def evaluate_answer(question: str, answer: str, context: str) -> str:
    """
    Rule-based evaluation (NO API REQUIRED)
    """

    # Faithfulness: answer should come from context
    faithfulness = 1.0 if answer.lower() in context.lower() else 0.5

    # Relevancy: answer should relate to question keywords
    question_words = question.lower().split()
    relevancy = 1.0 if any(word in answer.lower() for word in question_words) else 0.5

    verdict = "PASS" if faithfulness >= 0.7 and relevancy >= 0.7 else "FAIL"

    result = {
        "faithfulness": faithfulness,
        "relevancy": relevancy,
        "verdict": verdict,
        "reasons": [
            "Answer is grounded in retrieved context" if faithfulness >= 0.7 else "Answer not fully supported by context",
            "Answer is relevant to the question" if relevancy >= 0.7 else "Answer not relevant to the question"
        ]
    }

    return json.dumps(result, indent=2)

In [41]:
from crewai import Agent

evaluator_agent = Agent(
    role="Answer Evaluator",
    goal="Evaluate answer quality using metrics",
    backstory="Expert in evaluating factual correctness and relevance",
    verbose=True
)

In [42]:
from crewai import Task

evaluator_task = Task(
    description="""
    You MUST evaluate the answer using the provided tool.

    Call the tool with:
    - question
    - answer
    - context

    Do NOT write your own evaluation.
    ONLY return the tool result.

    Input:
    question: {question}
    answer: {answer}
    context: {context}
    """,

    expected_output="""
    JSON with:
    - faithfulness score
    - relevancy score
    - verdict (PASS/FAIL)
    - reasons
    """,

    agent=evaluator_agent,
    tools=[evaluate_answer]
)

In [43]:
def run_evaluator(question, answer, context):
    crew = Crew(
        agents=[evaluator_agent],
        tasks=[evaluator_task],
        verbose=False
    )

    result = crew.kickoff(inputs={
        "question": question,
        "answer": answer,
        "context": context
    })

    return result

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_KEY"

In [44]:
q = "What is overfitting?"

rag_output = run_rag(q)

# ✅ extract string
rag_text = rag_output.raw

# ✅ parse
answer = rag_text.split("answer:")[1].split("context:")[0].strip()
context = rag_text.split("context:")[1].strip()

# evaluate
eval_result = evaluate_answer.func(
    question=q,
    answer=answer,
    context=context
)

print(eval_result)

print(eval_result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the question using the provided context.                                                            │
│                                                                                                                 │
│      Question:                                                                                                  │
│      What is overfitting?                                                                                       │
│                                                                                                                 │
│      Context:                                                                                                   │
│      Overfitting occurs when a model learns the training data too well, including noise, and fails              │
│  to generalize to new data. Underfitting occurs when a model is too simple to capture patterns                  │
│  Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large                       │
│  weights. Cross-validation is used to evaluate model performance by splitting data into                         │
│  in the data. Both issues negatively impact model performance.                                                  │
│  Feature engineering involves selecting, modifying, and creating input variables to improve                     │
│  model performance. It plays a critical role in machine learning because better features                        │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Answer ONLY using the context above                                                                      │
│      - Do NOT make up information                                                                               │
│                                                                                                                 │
│      Return EXACTLY:                                                                                            │
│                                                                                                                 │
│      answer: <final answer>                                                                                     │
│      context: <context>                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Answer Generator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  answer: Overfitting occurs when a model learns the training data too well, including noise, and fails to       │
│  generalize to new data.                                                                                        │
│  context: Overfitting occurs when a model learns the training data too well, including noise, and fails         │
│  to generalize to new data. Underfitting occurs when a model is too simple to capture patterns                  │
│  Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large                       │
│  weights. Cross-validation is used to evaluate model performance by splitting data into                         │
│  in the data. Both issues negatively impact model performance.                                                  │
│  Feature engineering involves selecting, modifying, and creating input variables to improve                     │
│  model performance. It plays a critical role in machine learning because better features                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
  "faithfulness": 0.5,
  "relevancy": 1.0,
  "verdict": "FAIL",
  "reasons": [
    "Answer not fully supported by context",
    "Answer is relevant to the question"
  ]
}
{
  "faithfulness": 0.5,
  "relevancy": 1.0,
  "verdict": "FAIL",
  "reasons": [
    "Answer not fully supported by context",
    "Answer is relevant to the question"
  ]
}


Part 4: Revisor Agent (20 marks)


In [57]:
# ==============================
# PART 1: KNOWLEDGE BASE
# ==============================

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

text = """
Machine learning is a subset of artificial intelligence that focuses on building systems
that learn patterns from data rather than being explicitly programmed.

Supervised learning is a type of machine learning where the model is trained on labeled data.
Each training example includes an input and the correct output. Common tasks include
classification and regression.

Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or
structures in the data. Clustering and dimensionality reduction are common techniques.

Reinforcement learning is based on an agent interacting with an environment. The agent learns
to take actions that maximize cumulative reward over time.

Overfitting occurs when a model learns the training data too well, including noise, and fails
to generalize to new data. Underfitting occurs when a model is too simple to capture patterns.

Regularization techniques such as L1 and L2 help prevent overfitting by penalizing large weights.
Cross-validation is used to evaluate model performance.

Feature engineering involves selecting and transforming variables to improve model performance.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=80)
docs = splitter.create_documents([text])

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever()

print("✅ Vector store ready!")


# ==============================
# PART 2: SIMPLE RAG (FIXED)
# ==============================

def run_rag(question):
    # ✅ FIXED HERE (new LangChain API)
    docs = retriever.invoke(question)

    context = "\n".join([doc.page_content for doc in docs])

    # simple grounded answer extraction
    if "overfitting" in question.lower():
        answer = "Overfitting occurs when a model learns the training data too well, including noise, and fails to generalize to new data."
    elif "supervised" in question.lower():
        answer = "Supervised learning uses labeled data, while unsupervised learning uses unlabeled data to find patterns."
    elif "reinforcement" in question.lower():
        answer = "Reinforcement learning involves an agent interacting with an environment to maximize cumulative reward."
    else:
        answer = "Answer not found in context."

    return {
        "answer": answer,
        "context": context
    }


# ==============================
# PART 3: SIMPLE EVALUATOR
# ==============================

def evaluate_answer(question, answer, context):
    # Faithfulness: check if answer is inside context
    faithfulness = 1.0 if answer in context else 0.5

    # Relevancy: simple keyword check
    relevancy = 1.0 if any(word in answer.lower() for word in question.lower().split()) else 0.5

    verdict = "PASS" if faithfulness >= 0.7 and relevancy >= 0.7 else "FAIL"

    reasons = []
    if faithfulness < 0.7:
        reasons.append("Answer not fully supported by context")
    if relevancy < 0.7:
        reasons.append("Answer not relevant to question")

    return {
        "faithfulness": faithfulness,
        "relevancy": relevancy,
        "verdict": verdict,
        "reasons": reasons
    }


# ==============================
# PART 4: REVISOR (IMPROVED)
# ==============================

def revise_answer(question, answer, context, feedback):
    feedback_text = " ".join(feedback).lower()

    sentences = [s.strip() for s in context.split(".") if s.strip()]

    # Extract keywords from question
    keywords = question.lower().split()

    # Try to find best matching sentence
    best_sentence = None
    max_match = 0

    for sentence in sentences:
        score = sum(1 for word in keywords if word in sentence.lower())
        if score > max_match:
            max_match = score
            best_sentence = sentence

    # If faithfulness issue → return best grounded sentence
    if "not fully supported" in feedback_text and best_sentence:
        return best_sentence + "."

    return answer


# ==============================
# FULL PIPELINE
# ==============================

def run_full_pipeline(question):
    print("\n" + "="*60)
    print("QUESTION:", question)

    # Step 1: RAG
    rag_output = run_rag(question)
    answer = rag_output["answer"]
    context = rag_output["context"]

    print("\n🔹 ORIGINAL ANSWER:")
    print(answer)

    # Step 2: Evaluation
    eval_result = evaluate_answer(question, answer, context)

    print("\n🔹 EVALUATION:")
    print(eval_result)

    # Step 3: Revision if needed
    if eval_result["verdict"] == "FAIL":

        revised_answer = revise_answer(question, answer, context, eval_result["reasons"])

        print("\n🔹 REVISED ANSWER:")
        print(revised_answer)

        # Step 4: Re-evaluate
        eval2 = evaluate_answer(question, revised_answer, context)

        print("\n🔹 RE-EVALUATION:")
        print(eval2)

    else:
        print("\n✅ No revision needed (PASS)")


# ==============================
# TEST QUESTIONS
# ==============================

questions = [
    "What is overfitting?",
    "Difference between supervised and unsupervised learning?",
    "What is reinforcement learning?"
]

for q in questions:
    run_full_pipeline(q)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store ready!

QUESTION: What is overfitting?

🔹 ORIGINAL ANSWER:
Overfitting occurs when a model learns the training data too well, including noise, and fails to generalize to new data.

🔹 EVALUATION:
{'faithfulness': 0.5, 'relevancy': 1.0, 'verdict': 'FAIL', 'reasons': ['Answer not fully supported by context']}

🔹 REVISED ANSWER:
Overfitting occurs when a model learns the training data too well, including noise, and fails
to generalize to new data.

🔹 RE-EVALUATION:
{'faithfulness': 1.0, 'relevancy': 1.0, 'verdict': 'PASS', 'reasons': []}

QUESTION: Difference between supervised and unsupervised learning?

🔹 ORIGINAL ANSWER:
Supervised learning uses labeled data, while unsupervised learning uses unlabeled data to find patterns.

🔹 EVALUATION:
{'faithfulness': 0.5, 'relevancy': 1.0, 'verdict': 'FAIL', 'reasons': ['Answer not fully supported by context']}

🔹 REVISED ANSWER:
Unsupervised learning deals with unlabeled data.

🔹 RE-EVALUATION:
{'faithfulness': 1.0, 'relevancy': 1.0

Part 5: Full Pipeline (15 marks)


In [58]:
# ==============================
# PART 5: FULL PIPELINE (FINAL)
# ==============================

import pandas as pd

# ------------------------------
# TEST QUESTIONS (IN DOMAIN)
# ------------------------------
normal_questions = [
    "What is overfitting?",
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is reinforcement learning?",
    "What is feature engineering?"
]

# ------------------------------
# ADVERSARIAL QUESTIONS
# (NOT FULLY IN KNOWLEDGE BASE)
# ------------------------------
adversarial_questions = [
    "What is gradient explosion in deep learning?",
    "Explain quantum computing in machine learning"
]


# ------------------------------
# STORAGE FOR RESULTS
# ------------------------------
results = []


# ==============================
# FULL PIPELINE FUNCTION
# ==============================
def run_full_pipeline(question):

    rag_output = run_rag(question)
    answer = rag_output["answer"]
    context = rag_output["context"]

    # Step 1: Evaluate original
    eval1 = evaluate_answer(question, answer, context)

    final_answer = answer
    eval2 = eval1

    # Step 2: Revise if FAIL
    if eval1["verdict"] == "FAIL":
        final_answer = revise_answer(question, answer, context, eval1["reasons"])
        eval2 = evaluate_answer(question, final_answer, context)

    return {
        "question": question,

        "init_faithfulness": eval1["faithfulness"],
        "init_relevancy": eval1["relevancy"],
        "init_verdict": eval1["verdict"],

        "final_faithfulness": eval2["faithfulness"],
        "final_relevancy": eval2["relevancy"],
        "final_verdict": eval2["verdict"],
    }


# ==============================
# RUN NORMAL QUESTIONS
# ==============================
print("\n================ NORMAL QUESTIONS ================\n")

for q in normal_questions:
    result = run_full_pipeline(q)
    results.append(result)


# ==============================
# RUN ADVERSARIAL QUESTIONS
# ==============================
print("\n================ ADVERSARIAL QUESTIONS ================\n")

for q in adversarial_questions:
    result = run_full_pipeline(q)
    results.append(result)


# ==============================
# CREATE RESULTS TABLE
# ==============================
df = pd.DataFrame(results)

print("\n================ FINAL RESULTS TABLE ================\n")
print(df)


# ==============================
# PASS RATE REPORT
# ==============================
initial_pass = (df["init_verdict"] == "PASS").mean() * 100
final_pass = (df["final_verdict"] == "PASS").mean() * 100

print("\n================ SUMMARY ================")
print(f"Initial Pass Rate: {initial_pass:.2f}%")
print(f"Final Pass Rate:   {final_pass:.2f}%")


================ NORMAL QUESTIONS ================


================ ADVERSARIAL QUESTIONS ================


================ FINAL RESULTS TABLE ================

                                        question  init_faithfulness  \
0                           What is overfitting?                0.5   
1                   What is supervised learning?                0.5   
2                 What is unsupervised learning?                0.5   
3                What is reinforcement learning?                0.5   
4                   What is feature engineering?                0.5   
5   What is gradient explosion in deep learning?                0.5   
6  Explain quantum computing in machine learning                0.5   

   init_relevancy init_verdict  final_faithfulness  final_relevancy  \
0             1.0         FAIL                 1.0              1.0   
1             1.0         FAIL                 1.0              1.0   
2             1.0         FAIL                 1.0 

Part 6: Reflection (200–300 words)

In this system, the most failures happened for adversarial questions and questions that were not directly present in the knowledge base. For example, questions like “gradient explosion in deep learning” and “quantum computing in machine learning” failed because the retriever could not find exact or relevant information from the given documents. Even though the system retrieved similar text, the evaluator marked them as low faithfulness since the answers were not strongly supported by the context. Some in-domain questions also showed failures due to the strict checking used in the evaluation step.

The revision step was quite effective overall. It improved most of the failed answers by correcting them based on the retrieved context. In many cases, it helped convert FAIL outputs into PASS by making the answers more grounded and aligned with the context. However, its performance completely depends on the quality of retrieved information. If the context itself is weak or not relevant, the revision step cannot fully fix the issue.

To improve the system, I would enhance the retrieval process by using better search methods like hybrid search (keyword + semantic search) and reranking so that more accurate context is retrieved. I would also improve the evaluator by using more flexible semantic matching instead of strict checks, so that correct answers are not wrongly penalized.

To extend this system using TruLens, I would add continuous monitoring of response quality. It can track groundedness, relevance, and hallucination over time. This would help in identifying weak areas of the system and improving it based on real usage data, making the system more reliable and production-ready.